[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Claudia-Rocha-H/NATICUSTest/blob/main/notebooks/02_random_forest.ipynb)

Abrir este notebook en Google Colab para ejecutar el modelo de Random Forest de forma autónoma.

# 02. Random Forest NATICUSdroid



In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/Claudia-Rocha-H/NATICUSTest.git'
REPO_DIR = Path('/content/NATICUSTest')

import numpy as np  
import pandas as pd  

from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score  # noqa: F401
from sklearn.model_selection import GridSearchCV, StratifiedKFold 

_bootstrap_dependencies = (
    np,
    pd,
    RandomForestClassifier,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    GridSearchCV,
    StratifiedKFold,
)
print(f'Bootstrap imports loaded: {len(_bootstrap_dependencies)}')

def is_colab() -> bool:
    return 'COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules

def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'src' / 'prepare_data.py').exists():
            return candidate
    return REPO_DIR if is_colab() else Path.cwd()

repo_root = find_repo_root()
if is_colab() and not (repo_root / 'src' / 'prepare_data.py').exists():
    if not repo_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_root)], check=True)
    else:
        print(f'Repository directory already exists at {repo_root}.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Repository root: {repo_root}')
print('Imports listos para el entrenamiento del Random Forest.')

## Preprocesado

In [ ]:
from src.prepare_data import prepare_dataset

results = prepare_dataset()
summary = results['summary']
artifacts = results['artifacts']
X_train = results['X_train']
X_test = results['X_test']
y_train = results['y_train']
y_test = results['y_test']

print('Resumen del dataset')
print(summary)
print('')
print('Artefactos generados')
for name, value in artifacts.items():
    print(f'{name}: {value}')

print('')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

## Resumen  del dataset

In [ ]:
import matplotlib.pyplot as plt

train_df = X_train.copy()
train_df['Result'] = y_train.values

summary_table = pd.DataFrame([
    ['Filas', summary['rows']],
    ['Columnas totales', summary['columns']],
    ['Variables de entrada', summary['feature_columns']],
    ['Valores faltantes', summary['missing_values']],
    ['Clase 0', summary['class_distribution']['0']],
    ['Clase 1', summary['class_distribution']['1']],
], columns=['Métrica', 'Valor'])

print(summary_table.to_string(index=False))
print('')
print('Primeras 10 filas del conjunto de entrenamiento')
print(train_df.head(10).to_string(index=False))
print('')
print('Últimas 10 filas del conjunto de entrenamiento')
print(train_df.tail(10).to_string(index=False))
print('')
sample_size = min(5, len(train_df))
print(f'Muestra aleatoria de {sample_size} filas')
print(train_df.sample(sample_size, random_state=42).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = pd.Series(summary['class_distribution']).astype(int)
class_counts.index = ['Benigna (0)', 'Maliciosa (1)']
class_counts.plot(kind='bar', ax=axes[0], color=['#4C78A8', '#E45756'])
axes[0].set_title('Distribución de clases')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Número de aplicaciones')
axes[0].tick_params(axis='x', rotation=0)

feature_prevalence = X_train.mean().sort_values(ascending=False).head(10)
axes[1].barh(feature_prevalence.index[::-1], feature_prevalence.values[::-1], color='#72B7B2')
axes[1].set_title('Top 10 permisos más frecuentes en entrenamiento')
axes[1].set_xlabel('Proporción de presencia')
axes[1].set_ylabel('Permiso')

plt.tight_layout()
plt.show()
